# Prompt 10 — FINAL corrected baseline validation

**Purpose:** validate the 600 scored responses, repair the experiment-specific ID collision, independently check NLL arithmetic, and optionally audit actual saved rows. This notebook **does not evaluate test-set performance**.

**Upload:** `AITR_Prompt09B_scores.zip` **or** `AITR_Prompt11_repaired_scores.zip`. Optionally upload `AITR_Prompt09_data.zip` in the second upload cell to audit actual question/answer alignment. Run cells in order on Colab. All claims in the exported report are based on checks actually run.

In [ ]:
import io, json, zipfile, hashlib, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from google.colab import files
from transformers import AutoTokenizer, AutoModelForCausalLM
print('=== ENVIRONMENT ===')
print('torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
print('Upload either AITR_Prompt09B_scores.zip OR AITR_Prompt11_repaired_scores.zip')
uploaded = files.upload()
zipnames = [x for x in uploaded if x.lower().endswith('.zip')]
assert len(zipnames) == 1, 'Upload exactly one scoring ZIP.'
source_zip = zipnames[0]
frames = {}
with zipfile.ZipFile(source_zip) as z:
    for split, expected in [('train',420),('validation',90),('test',90)]:
        matches = [x for x in z.namelist() if x == f'{split}_scores.csv' or x.endswith('/'+f'{split}_scores.csv')]
        assert len(matches) == 1, f'Expected exactly one {split}_scores.csv, found {matches}'
        frames[split] = pd.read_csv(io.BytesIO(z.read(matches[0])))
        assert len(frames[split]) == expected
        print(split, 'rows:', len(frames[split]), 'labels:', frames[split].label.value_counts().sort_index().to_dict())
print('=== SCORE FILES LOADED: PASS ===')

In [ ]:
# Repair synthetic row IDs if necessary; never pretend these are original HaluEval IDs.
original_id_sets = [set(x['ID'].astype(str)) for x in frames.values()]
collision = bool(original_id_sets[0]&original_id_sets[1] or original_id_sets[0]&original_id_sets[2] or original_id_sets[1]&original_id_sets[2])
print('Overlapping input IDs:', collision)
repaired = {}
for split, df in frames.items():
    x = df.copy()
    assert 'row_index' in x.columns
    row_nums = pd.to_numeric(x.row_index, errors='raise').astype(int)
    assert row_nums.is_unique and set(row_nums) == set(range(len(x))), f'{split}: invalid row_index'
    if collision:
        x['ID'] = split + '_' + row_nums.astype(str)
    repaired[split] = x
all_ids = pd.concat([x.ID.astype(str) for x in repaired.values()], ignore_index=True)
assert all_ids.is_unique and len(all_ids) == 600
print('=== ID INTEGRITY: PASS ===')
print('Identifier type:', 'new split-specific row IDs' if collision else 'input IDs retained')

In [ ]:
required = ['label','status','mean_nll','p90_nll','answer_token_length','context_tokens_dropped','boundary_crossing_tokens']
report = {'model':'distilgpt2','source_scoring_zip':source_zip,
          'identifier_note':'Split-specific row IDs created if original scoring IDs collided; these are not HaluEval source IDs.',
          'splits':{},'checks':{}}
for split,df in repaired.items():
    assert set(required).issubset(df.columns), f'{split}: missing required columns'
    assert df.status.eq('ok').all(), f'{split}: scoring failures'
    nums = df[['mean_nll','p90_nll','answer_token_length','context_tokens_dropped','boundary_crossing_tokens']].apply(pd.to_numeric, errors='coerce')
    assert np.isfinite(nums.to_numpy()).all(), f'{split}: nonfinite values'
    assert (nums.to_numpy() >= 0).all(), f'{split}: negative values'
    assert (nums.answer_token_length > 0).all()
    counts = df.label.value_counts().sort_index().to_dict()
    assert set(counts) == {0,1}, f'{split}: missing label class'
    report['splits'][split] = {'rows':len(df), 'label_counts':{str(k):int(v) for k,v in counts.items()},
        'mean_answer_token_length':float(nums.answer_token_length.mean()),
        'boundary_crossing_rows':int((nums.boundary_crossing_tokens>0).sum()),
        'context_trimmed_rows':int((nums.context_tokens_dropped>0).sum())}
    print(f'=== {split.upper()} AUDIT: PASS ===',report['splits'][split])
report['checks']['score_integrity']='PASS'
report['checks']['id_integrity']='PASS'
print('=== SCORE INTEGRITY: PASS ===')

## Independent arithmetic audit

This recomputes per-token losses by **two distinct arithmetic routes** on a synthetic example. It tests the calculation, not the correctness of all 600 saved scores.

In [ ]:
tok = AutoTokenizer.from_pretrained('distilgpt2', use_fast=True)
model = AutoModelForCausalLM.from_pretrained('distilgpt2').to('cuda' if torch.cuda.is_available() else 'cpu').eval()
question='What is the capital of France?'
answer='Paris is the capital of France.'
prompt=f'Question: {question}\nAnswer: '
enc=tok(prompt+answer, add_special_tokens=False, return_offsets_mapping=True)
positions=[i for i,(s,e) in enumerate(enc['offset_mapping']) if s>=len(prompt) and e>s and i>0]
assert positions
ids=torch.tensor([enc['input_ids']], device=next(model.parameters()).device)
with torch.inference_mode():
    logits=model(ids).logits[0,:-1,:].float()
    targets=ids[0,1:]
    ce=torch.nn.functional.cross_entropy(logits,targets,reduction='none')
    lp=torch.log_softmax(logits,dim=-1)
    independent=-lp.gather(1,targets.unsqueeze(1)).squeeze(1)
    selected=torch.tensor([i-1 for i in positions],device=ids.device)
    a=ce.index_select(0,selected).cpu().numpy()
    b=independent.index_select(0,selected).cpu().numpy()
assert np.allclose(a,b,rtol=1e-5,atol=1e-5)
print('=== INDEPENDENT SMOKE AUDIT: PASS ===')
print('token count:',len(a),'mean NLL:',float(a.mean()),'p90 NLL:',float(np.percentile(a,90)))
report['checks']['independent_arithmetic_smoke']='PASS'
report['checks']['smoke_max_abs_difference']=float(np.max(np.abs(a-b)))

## Optional stronger check: original data ZIP

Upload `AITR_Prompt09_data.zip` if available. We compare the original label ordering and independently recompute three scores per split (nine total). If you skip this, the report will explicitly say that original-row validation was not performed.

In [ ]:
print('OPTIONAL: Upload AITR_Prompt09_data.zip, or click Cancel/skip upload if unavailable.')
print('To skip this step without upload, set data_zip = None in the next cell.')
# Uncomment to upload when you have the data ZIP:
# data_uploaded = files.upload()
# data_zip = next((x for x in data_uploaded if x.lower().endswith('.zip')), None)
data_zip = None

In [ ]:
if data_zip is None:
    print('=== ORIGINAL ROW AUDIT: NOT PERFORMED (optional data ZIP not supplied) ===')
    report['checks']['original_row_audit']='NOT PERFORMED'
else:
    with zipfile.ZipFile(data_zip) as z:
        original={}
        for split in ['train','validation','test']:
            matches=[x for x in z.namelist() if x==f'{split}.csv' or x.endswith('/'+f'{split}.csv')]
            assert len(matches)==1
            original[split]=pd.read_csv(io.BytesIO(z.read(matches[0])))
    checks=[]
    for split, src in original.items():
        scored=repaired[split].sort_values('row_index').reset_index(drop=True)
        assert len(src)==len(scored)
        assert np.array_equal(src.label.astype(int).to_numpy(),scored.label.astype(int).to_numpy())
        for idx in sorted(set([0,len(src)//2,len(src)-1])):
            q=str(src.iloc[idx].user_query); ans=str(src.iloc[idx].chatgpt_response)
            p=f'Question: {q}\nAnswer: '
            enc=tok(p+ans,add_special_tokens=False,return_offsets_mapping=True,truncation=False)
            pos=[i for i,(s,e) in enumerate(enc['offset_mapping']) if s>=len(p) and e>s]
            max_ctx=int(model.config.n_positions)
            assert len(pos)<max_ctx
            trim=max(0,len(enc['input_ids'])-max_ctx)
            ids_trim=enc['input_ids'][trim:]
            pos=[i-trim for i in pos if i>=trim and i-trim>0]
            assert pos
            inp=torch.tensor([ids_trim],device=next(model.parameters()).device)
            with torch.inference_mode():
                logits=model(inp).logits[0,:-1,:].float()
                loss=torch.nn.functional.cross_entropy(logits,inp[0,1:],reduction='none')
                vals=loss[torch.tensor([i-1 for i in pos],device=inp.device)].cpu().numpy()
            rec=scored.iloc[idx]
            assert np.isclose(vals.mean(),rec.mean_nll,atol=1e-4,rtol=1e-4)
            assert np.isclose(np.percentile(vals,90),rec.p90_nll,atol=1e-4,rtol=1e-4)
            assert len(vals)==int(rec.answer_token_length)
            checks.append((split,idx))
    print('=== ORIGINAL ROW AUDIT: PASS ===',checks)
    report['checks']['original_row_audit']='PASS: 3 recomputed rows per split (9 total)'


In [ ]:
out=Path('/content/AITR_Prompt10_FINAL');out.mkdir(exist_ok=True)
for split,df in repaired.items():df.to_csv(out/f'{split}_scores.csv',index=False)
(out/'baseline_validation.json').write_text(json.dumps(report,indent=2))
lines=['# Prompt 10 — Final Baseline Validation','',
       '## Scope','Score-file integrity, ID repair, independent NLL arithmetic smoke test, optional original-row recomputation. No predictive performance evaluated.','',
       '## Verified checks']
for k,v in report['checks'].items():lines.append(f'- {k}: {v}')
lines+=['','## Data splits']
for k,v in report['splits'].items():lines.append(f'- {k}: {v["rows"]} rows; labels {v["label_counts"]}; boundary-crossing rows {v["boundary_crossing_rows"]}.')
lines+=['','## Limitations','- Split-specific row IDs are not original HaluEval source IDs.',
        '- Question-overlap audit is supported by the earlier Prompt 09A run, not independently rechecked using score-only files.',
        '- The first token crossing the prompt/answer boundary was excluded by the scoring protocol.',
        '- DistilGPT-2 is a surrogate scorer, not the model that generated the labelled responses.',
        '- Full 600-row independent recomputation was not performed.']
(out/'baseline_validation.md').write_text('\n'.join(lines)+'\n')
zip_path='/content/AITR_Prompt10_FINAL.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(out.iterdir()):z.write(f,arcname=f.name)
print('=== PROMPT 10 FINAL ===')
print('Score integrity:',report['checks']['score_integrity'])
print('ID integrity:',report['checks']['id_integrity'])
print('Arithmetic audit:',report['checks']['independent_arithmetic_smoke'])
print('Original row audit:',report['checks']['original_row_audit'])
print('Saved:',zip_path)
files.download(zip_path)